# С оптимизациями

In [ ]:
import sys, os
sys.path.append('..')

In [ ]:
from pymatgen.core import Structure, Lattice, Element
import numpy as np

def atomicdata_to_pmg_structure(atomic_data):
    cell = np.array(atomic_data.cell[0], copy=False)
    positions = np.array(atomic_data.pos, copy=False)
    atomic_numbers = np.array(atomic_data.atomic_numbers, copy=False)
    elements = [Element.from_Z(z).symbol for z in atomic_numbers]
    lattice = Lattice(cell)
    structure = Structure(
        lattice=lattice,
        species=elements,
        coords=positions,
        coords_are_cartesian=True
    )
    return structure

In [ ]:
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ltol=0.3
stol=0.5
angle_tol=5

matcher = StructureMatcher(
        ltol=ltol,
        stol=stol,
        angle_tol=angle_tol,
        primitive_cell=True,
        scale=False
    )

def compare_structures(s1, s2, to_print=False):
    if s1.composition != s2.composition:
        return False

    tol = ltol
    for a, b in zip(s1.lattice.abc, s2.lattice.abc):
        if abs(a-b)/max(a,b) > tol:
            if to_print: print(f"Lattice lengths differ by more than {int(ltol * 100)}%:", s1.lattice.abc, s2.lattice.abc)
            return False
    for α, β in zip(s1.lattice.angles, s2.lattice.angles):
        if abs(α-β) > angle_tol:
            if to_print: print("Lattice angles differ by more than {angle_tol}°:", s1.lattice.angles, s2.lattice.angles)
            return False

    are_fit = matcher.fit(s1.get_primitive_structure(), s2.get_primitive_structure())

    if to_print:
        if are_fit:
            print("Structures match (within tolerances) 🎉")
        else:
            print("Structures do not match.")
    return are_fit

In [4]:
from fairchem.core.datasets import AseDBDataset
from tqdm import tqdm
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *
from slices.core import SLICES
import traceback

folder = "../data/raw/sAlex/train"

def convert_to_slices(dataset_folder, save_file, max_amount=400_000):
    config_kwargs = {}
    dataset_paths = [
        dataset_folder
    ]
    dataset = AseDBDataset(config=dict(src=dataset_paths, **config_kwargs))

    backend = SLICES(relax_model="chgnt", steps=100)
    # backend = SLICES()

    overall_size = len(dataset)
    errors = 0
    no_match = 0
    no_fit = 0

    results = []

    with tqdm(total=max_amount, smoothing=0.05) as pbar:
        with open(save_file, "a") as f:
            for i, row in enumerate(dataset):
                try:
                    energy = row.energy.item() if hasattr(row.energy, 'item') else row.energy
                    natoms = row.natoms.item() if hasattr(row.natoms, 'item') else row.natoms
                    e_per_atom = energy / natoms
                    if e_per_atom < -0.5 and natoms >= 1 and natoms < 20:
                        no_match += 1
                        pbar.set_postfix(passed=f"{i}/{overall_size}", pass_rate=f"{len(results)/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
                        continue
                    original_structure = atomicdata_to_pmg_structure(row)
                    slices = backend.structure2SLICES(original_structure, strategy=3)
                    reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices, strategy=3)
                    is_okay = compare_structures(original_structure, reconstructed_structure)
                    if not is_okay:
                        no_fit += 1
                        pbar.set_postfix(passed=f"{i}/{overall_size}", pass_rate=f"{len(results)/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
                        continue
                    f.write(f"{slices};{natoms};{e_per_atom}\n")
                    f.flush()
                    results.append({"slices": slices, "natoms": natoms, "e_per_atoms": e_per_atom})
                    pbar.update(1)
                    pbar.set_postfix(passed=f"{i}/{overall_size}", pass_rate=f"{len(results)/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
                    if len(results) >= max_amount:
                        print(f"{max_amount} samples found!")
                        break
                except Exception as e:
                    errors += 1
                    # print("Error:", e)
                    # traceback.print_exc()

    print("Saving samples...")

    df = pd.DataFrame(data)
    try:
        df.to_csv(save_file, index=False)
    except Exception as e:
        print(e)
    finally:
        return df

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [5]:
def filtered_range(start, stop, drop_list):
    drop_set = set(drop_list)
    for x in range(start, stop):
        if x not in drop_set:
            yield x

def chunked(iterable, chunk_size):
    chunk = []
    for item in iterable:
        chunk.append(item)
        if len(chunk) == chunk_size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk

In [6]:
import multiprocessing as mp
import psutil
import time
import os

def run_with_memory_limit(target, mem_frac=0.3, poll_interval=0.5, *args, **kwargs):
    ctx = mp.get_context('spawn')
    queue = ctx.Queue()
    proc = ctx.Process(target=target, args=(queue, *args), kwargs=kwargs)
    proc.start()

    ps_proc = psutil.Process(proc.pid)
    total_mem = psutil.virtual_memory().total

    result = None
    killed = False

    while proc.is_alive():
        try:
            mem = ps_proc.memory_info().rss
            if mem / total_mem > mem_frac:
                print(f"Killing process {proc.pid}: using {mem/total_mem:.2%} RAM")
                proc.terminate()
                killed = True
                break
        except psutil.NoSuchProcess:
            break
        time.sleep(poll_interval)

    if not killed and not queue.empty():
        result = queue.get()
    proc.join()
    if killed:
        return Exception("Memory exceeded")
    return result

In [9]:
from fairchem.core.datasets import AseDBDataset
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor   
from tqdm import tqdm
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *
from slices.core import SLICES
import traceback
from multiprocessing import Queue, Process

folder = "../data/raw/sAlex/train"

def process_rows_job(dataset_folder, receiver_range, done_queue, done_processes_queue):
    sys.stdout = open(os.devnull, 'w')
    sys.stderr = open(os.devnull, 'w')

    config_kwargs = {}
    dataset_paths = [
        dataset_folder
    ]
    df = AseDBDataset(config=dict(src=dataset_paths, **config_kwargs))

    # backend = SLICES(relax_model="chgnt", steps=100)
    backend = SLICES(relax_model="chgnet", steps=100)

    for i in receiver_range:
        row = df[i]
        try:
            energy = row.energy.item() if hasattr(row.energy, 'item') else row.energy
            natoms = row.natoms.item() if hasattr(row.natoms, 'item') else row.natoms
            e_per_atom = energy / natoms
            if e_per_atom >= -0.5 or natoms >= 20:
                done_queue.put((0, None, i))
                continue
            # if e_per_atom < -0.5 and natoms >= 1 and natoms < 20:
            #     done_queue.put((0, None, i))
            #     # no_match += 1
            #     # pbar.set_postfix(passed=f"{i}/{overall_size}", pass_rate=f"{len(results)/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
            #     continue
            original_structure = atomicdata_to_pmg_structure(row)
            slices = backend.structure2SLICES(original_structure, strategy=3)
            reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices, strategy=3)
            is_okay = compare_structures(original_structure, reconstructed_structure)
            if not is_okay:
                reconstructed_structure, final_energy_per_atom = backend.relax(reconstructed_structure)
                # reconstructed_structure, final_energy_per_atom = run_with_memory_limit(get_relax, slices=slices_NdSiRu)
                is_okay = compare_structures(original_structure, reconstructed_structure)
            if not is_okay:
                done_queue.put((1, None, i))
                # no_fit += 1
                # pbar.set_postfix(passed=f"{i}/{overall_size}", pass_rate=f"{len(results)/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
                continue
            # f.write(f"{slices};{natoms};{e_per_atom}\n")
            # f.flush()
            done_queue.put((2, f"{i};{slices};{natoms};{e_per_atom}", i))
            # results.append({"slices": slices, "natoms": natoms, "e_per_atoms": e_per_atom})
            # pbar.update(1)
            # pbar.set_postfix(passed=f"{i}/{overall_size}", pass_rate=f"{len(results)/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
            # if len(results) >= max_amount:
            #     print(f"{max_amount} samples found!")
            #     break
        except Exception as e:
            # errors += 1
            done_queue.put((3, None, i))
            # print("Error:", e)
            # traceback.print_exc()

def convert_to_slices_parallel(dataset_folder, save_file, max_amount=400_000, jobs=4):
    # backend = SLICES(relax_model="chgnt", steps=100)
    # backend = SLICES()

    config_kwargs = {}
    dataset_paths = [
        dataset_folder
    ]

    df = AseDBDataset(config=dict(src=dataset_paths, **config_kwargs))

    batch_size = len(df) // jobs
    if os.path.exists(f"save-idx-{save_file}"):
        with open(f"save-idx-{save_file}", "r") as f:
            drop = [int(line.strip()) for line in f if line.strip()]
            filtered = filtered_range(0, len(df), drop)
            batch_size = (len(df) - len(drop)) // jobs
            batches = list(chunked(filtered, batch_size))
    else:
        drop = []
        batches = [range(i * batch_size, (i + 1) * batch_size) for i in range(jobs - 1)]
        batches.append(range((jobs - 1) * batch_size, len(df)))

    overall_size = len(df)
    errors = 0
    no_match = 0
    no_fit = 0

    results = []

    result_queue = Queue()

    done_processes_queue = Queue()

    overall_size = len(df)
    errors = 0
    no_match = 0
    no_fit = 0
    passed_results = len(drop)

    # processes = []
    for i in range(jobs):
        p = Process(target=process_rows_job, args=(dataset_folder, batches[i], result_queue, done_processes_queue))
        # processes.append(p)
        p.start()

    if os.path.exists(save_file):
        with open(save_file, "r") as f:
            i = len([line for line in f if line.strip()])
    else:
        i = 0

    # try:
    # with ThreadPoolExecutor(max_workers=jobs) as executor:
    with ProcessPoolExecutor(max_workers=jobs) as executor:
        # for i in executor.map(process_rows_job, [dataset_folder] * jobs, batches, [result_queue] * jobs, [done_processes_queue] * jobs):
        #     print(i)
        with tqdm(total=max_amount, smoothing=0.05, initial=i) as pbar:
        # with tqdm(total=overall_size, smoothing=0.05, initial=passed_results) as pbar:
            with open(save_file, "a") as f:
                with open(f"save-idx-{save_file}", "a") as f_idx:
                    while done_processes_queue.qsize() < jobs:
                        status, results, idx = result_queue.get(block=True, timeout=None)
                        if results is not None:
                            i += 1
                            pbar.update(1)
                            f.write(results + "\n")
                            f.flush()
                        elif status == 0:
                            no_match += 1
                        elif status == 1:
                            no_fit += 1
                        else:
                            errors += 1
                        # pbar.update(1)
                        passed_results += 1
                        # pbar.set_postfix(passed=f"{i}/{max_amount}", pass_rate=f"{i/max_amount:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
                        pbar.set_postfix(passed=f"{passed_results}/{overall_size}", pass_rate=f"{passed_results/overall_size:.2%}", error=f"{errors}", no_match=f"{no_match}", no_fit=f"{no_fit}")
                        f_idx.write(str(idx) + "\n")
                        f_idx.flush()
    # except Exception as e:
    #     print(e)

In [ ]:
convert_to_slices_parallel("../data/raw/sAlex/train", "salex-benchmark-opt.csv", jobs=16)

  0%|          | 133/400000 [01:50<227:02:04,  2.04s/it, error=69, no_fit=0, no_match=5, pass_rate=0.00%, passed=228/10447765]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
  0%|          | 291/400000 [05:48<91:40:57,  1.21it/s, error=138, no_fit=0, no_match=25, pass_rate=0.00%, passed=475/10447765]